In [1]:
import pandas as pd
import plotly.express as px
import hashlib

In [2]:
# Ladda in datafilerna
df = pd.read_csv("../data/athlete_events.csv")
noc = pd.read_csv("../data/noc_regions.csv")

df.head()

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,NaN


In [3]:
# Anonymisera namnen med SHA-256
df_anonym = df.copy()

df_anonym['Name_hash'] = df_anonym['Name'].apply(lambda x: hashlib.sha256(x.encode()).hexdigest())

df_anonym = df_anonym.drop(columns=['Name'])
df_anonym.head()

,ID,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal,Name_hash
0,1,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN,3a4eef48434c66b3f14ab0221f6762d0ef7c6135ab2790...
1,2,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN,a6430cc6630934275dc6283f7e97e9625e6587cdddec7a...
2,3,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN,9c198b205332c2c8e1542e0f9534b9e270780a41d978ec...
3,4,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold,0a477bb1c5ad39716f9c775e54d18d16aa8b37ada55548...
4,5,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,NaN,5b7be356aa28178096dc6747f0b8e4e393eaceb5f95310...


In [4]:
# Filtera landet (Frankrike)
france_df = df_anonym[df_anonym['Team'] == 'France']
france_df.head()

,ID,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal,Name_hash
98,34,M,30.0,187.0,76.0,France,FRA,2012 Summer,2012,Summer,London,Athletics,"Athletics Men's 1,500 metres",NaN,f55d2538e2894e8989e9e4c980e655fceb9a26bc0206f7...
145,52,M,22.0,189.0,80.0,France,FRA,1976 Summer,1976,Summer,Montreal,Athletics,Athletics Men's Pole Vault,NaN,0a233c98a85ac1a48fbfea66cbdd0854f358ee489ccff0...
149,56,M,21.0,NaN,NaN,France,FRA,1956 Summer,1956,Summer,Melbourne,Cycling,"Cycling Men's Road Race, Individual",NaN,1b8c926b7a11c580396202a098140668ef85328a14050a...
150,56,M,21.0,NaN,NaN,France,FRA,1956 Summer,1956,Summer,Melbourne,Cycling,"Cycling Men's Road Race, Team",Gold,1b8c926b7a11c580396202a098140668ef85328a14050a...
173,73,M,23.0,182.0,86.0,France,FRA,2008 Summer,2008,Summer,Beijing,Handball,Handball Men's Handball,Gold,277d20d8e003ceb8c9e1e2e1c570b972e9d3ea94543378...


In [5]:
# Medaljer per sport
top_sports = (
    france_df.dropna(subset=['Medal'])
    .groupby('Sport')['Medal']
    .count()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

print(top_sports)

fig = px.bar(top_sports,
             x='Sport',
             y='Medal',
             title='Top 10 sporter där Frankrike fått flest medaljer')
fig.show()

           Sport  Medal
0        Fencing    310
1        Cycling    151
2      Athletics    110
3         Rowing     89
4       Handball     77
5  Equestrianism     77
6       Swimming     71
7       Shooting     69
8     Gymnastics     58
9       Canoeing     49


In [6]:
# Mrdaljer per år
medals_per_year = (
    france_df.dropna(subset=['Medal'])
    .groupby('Year')['Medal']
    .count()
    .reset_index()
)

print(medals_per_year)

fig = px.bar(medals_per_year,
              x='Year',
              y='Medal',
              title='Antal medaljer per OS (Frankrike)')
fig.show()

    Year  Medal
0   1896     11
1   1900     75
2   1904      1
3   1906     45
4   1908     36
5   1912     23
6   1920    134
7   1924    109
8   1928     46
9   1932     42
10  1936     45
11  1948     77
12  1952     41
13  1956     33
14  1960     15
15  1964     38
16  1968     36
17  1972     25
18  1976     21
19  1980     30
20  1984     70
21  1988     31
22  1992     65
23  1994     11
24  1996     49
25  1998     13
26  2000     64
27  2002     13
28  2004     53
29  2006     15
30  2008     77
31  2010     14
32  2012     78
33  2014     18
34  2016     96


In [7]:
france_df[france_df['Year'] == 1920]['Medal'].value_counts()

Medal
Silver    68
Bronze    55
Gold      11
Name: count, dtype: int64

In [8]:
# Ålderfördelnin
fig = px.histogram(france_df,
                   x='Age',
                   title='Ålderfördelning bland franska atleter')
fig.show()

In [9]:
# Fördelning av medaljtyper
medal_types = (
    france_df.dropna(subset=['Medal'])
    .groupby('Medal')
    .size()
    .reset_index(name='Antal')
)

fig = px.pie(medal_types,
             names='Medal',
             values='Antal',
             title='Fördelning av medaljtyper (Frankrike)')
fig.show()

In [10]:
unique_medals = (
    france_df.dropna(subset=['Medal'])
    .drop_duplicates(subset=['Team', 'Year', 'Sport', 'Event', 'Medal'])
)

### Sammanfattning 

Resultaten visar att Frankrike är särkilt framgångsrikt i sporter som **Fencing** och **Cycling**, och att de flesta atleterna är mellan **20-30 år gamla**.

Observera att antalet medaljer är baserat på individuella idrottare.
Det innebär att lagmedaljer (t.ex. fotball) räknas flera gånger - en per spelare.

Om man vill räkna unika medaljer (en medalj per lag och event),
kan man använda en variant av datan utan dubbletter, till exampel:

'unique_medals = (
    country_df.dropna(subset=['Medal'])
    .drop_duplicates(subset=['Team', 'Year', 'Sport', 'Event', 'Medal'])
)'